<a href="https://colab.research.google.com/github/Mromererg/vindr-spineXR-detection/blob/main/02_yolo_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO W&B Sweep - Hiperparametre Optimizasyonu

Bu notebook, VinDr-SpineXR veri seti üzerinde YOLO modeli için W&B Sweep kullanarak hiperparametre optimizasyonu yapar.

**Model:** YOLOv11l / YOLO26l  
**Sweep Yöntemi:** Bayesian Optimizasyon  
**Optimize Edilen Metrik:** mAP@0.5  
**Deneme Sayısı:** 50  
**Epoch:** 10 (sweep testi için)

**Optimize Edilen Hiperparametreler:**
- lr0, lrf (learning rate)
- momentum, weight_decay
- Augmentation: hsv_h, hsv_s, hsv_v, flipud, fliplr, mosaic, mixup, degrees, translate, scale
- batch_size

## 1. Kurulum

In [ ]:
!pip install ultralytics wandb -q

## 2. Google Drive Mount ve Dataset Hazırlama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

DATASET_DIR = '/content/spineXR_yolo_dataset_guncel'
BASE = '/content/drive/MyDrive/dataset/abnormal'
TRAIN_JSON = f'{BASE}/splits/original/train_coco_train.json'
VAL_JSON   = f'{BASE}/splits/original/train_coco_val.json'

os.makedirs(f'{DATASET_DIR}/images/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/val',   exist_ok=True)

with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(VAL_JSON) as f:
    val_data = json.load(f)

for img in train_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/train/{img['file_name']}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

for img in val_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/val/{img['file_name']}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

for f in os.listdir(f'{BASE}/yolo_labels/train'):
    src = f'{BASE}/yolo_labels/train/{f}'
    dst = f'{DATASET_DIR}/labels/train/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

for f in os.listdir(f'{BASE}/yolo_labels/val'):
    src = f'{BASE}/yolo_labels/val/{f}'
    dst = f'{DATASET_DIR}/labels/val/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
nc: 7
names:
  - Osteophytes
  - Spondylolysthesis
  - Disc space narrowing
  - Other lesions
  - Surgical implant
  - Foraminal stenosis
  - Vertebral collapse
"""
with open(f'{DATASET_DIR}/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('Train images:', len(os.listdir(f'{DATASET_DIR}/images/train')))
print('Val images:',   len(os.listdir(f'{DATASET_DIR}/images/val')))

## 3. W&B Login

In [ ]:
import wandb
wandb.login()  # API key girilecek

## 4. Sweep Fonksiyonu ve Konfigürasyon

In [ ]:
from ultralytics import YOLO
import wandb, os, shutil

YAML_PATH = '/content/spineXR_yolo_dataset_guncel/dataset.yaml'

def sweep_train():
    run = wandb.init()
    cfg = wandb.config

    # Her denemede temiz başlasın
    run_dir = f'/content/runs/detect/spineXR-yolo/{run.name}'
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir)

    model = YOLO('yolo11l.pt')  # yolo26l.pt ile de denenebilir

    model.train(
        data=YAML_PATH,
        epochs=cfg.epochs,
        imgsz=640,
        batch=cfg.batch_size,
        lr0=cfg.lr0,
        lrf=cfg.lrf,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
        optimizer='SGD',
        hsv_h=cfg.hsv_h,
        hsv_s=cfg.hsv_s,
        hsv_v=cfg.hsv_v,
        flipud=cfg.flipud,
        fliplr=cfg.fliplr,
        mosaic=cfg.mosaic,
        mixup=cfg.mixup,
        degrees=cfg.degrees,
        translate=cfg.translate,
        scale=cfg.scale,
        close_mosaic=5,
        patience=5,
        save=True,
        plots=False,
        verbose=False,
        project='spineXR-yolo',
        name=run.name,
        exist_ok=True,
    )

    run.finish()


# Sweep konfigürasyonu
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'metrics/mAP50(B)', 'goal': 'maximize'},
    'parameters': {
        'lr0':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'lrf':          {'distribution': 'uniform', 'min': 0.01, 'max': 0.2},
        'momentum':     {'distribution': 'uniform', 'min': 0.85, 'max': 0.98},
        'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-3},
        'hsv_h':        {'distribution': 'uniform', 'min': 0.0, 'max': 0.1},
        'hsv_s':        {'distribution': 'uniform', 'min': 0.0, 'max': 0.9},
        'hsv_v':        {'distribution': 'uniform', 'min': 0.0, 'max': 0.5},
        'flipud':       {'values': [0.0, 0.5]},
        'fliplr':       {'values': [0.0, 0.5]},
        'mosaic':       {'values': [0.0, 1.0]},
        'mixup':        {'values': [0.0, 0.2]},
        'degrees':      {'distribution': 'uniform', 'min': 0.0, 'max': 10.0},
        'translate':    {'distribution': 'uniform', 'min': 0.0, 'max': 0.2},
        'scale':        {'distribution': 'uniform', 'min': 0.0, 'max': 0.5},
        'batch_size':   {'values': [8, 16]},
        'epochs':       {'value': 10},
    }
}

print('Sweep konfigürasyonu hazır.')

## 5. Sweep Başlat

- **count=50**: 50 deneme yapılır
- Her deneme 10 epoch çalışır
- A100 GPU'da ~12 saat sürer

In [ ]:
sweep_id = wandb.sweep(sweep_config, project='spineXR-yolo')
wandb.agent(sweep_id, function=sweep_train, count=50)

## 6. Sweep Sonuçlarını İncele

In [ ]:
import wandb

api = wandb.Api()

# Sweep ID'yi buraya gir
SWEEP_ID = 'KULLANICI_ADI/spineXR-yolo/SWEEP_ID'

sweep = api.sweep(SWEEP_ID)
runs = sorted(sweep.runs, key=lambda r: r.summary.get('metrics/mAP50(B)', 0), reverse=True)

print('En iyi 10 run (mAP@0.5 sırasına göre):')
for r in runs[:10]:
    map50 = r.summary.get('metrics/mAP50(B)', 0)
    print(f"\nRun: {r.name}")
    print(f"  mAP@0.5: {map50:.4f}")
    print(f"  lr0: {r.config.get('lr0'):.6f}")
    print(f"  momentum: {r.config.get('momentum'):.4f}")
    print(f"  batch: {r.config.get('batch_size')}")
    print(f"  mosaic: {r.config.get('mosaic')}")
    print(f"  flipud: {r.config.get('flipud')}, fliplr: {r.config.get('fliplr')}")